# 03 — Two-Stage Retrieval: Cross-Encoder Reranking

**Track:** Intermediate · **Stage:** Retrieval Engineering  
**Scenario:** Atlas procurement and compliance teams search vendor, database, and Regulation R-17 evidence.

> **A reranker improves ordering of retrieved candidates. It cannot recover evidence that never entered the candidate set.**

This notebook evolves the original three-passage API demonstration into a controlled experiment: one difficult enterprise corpus, one labelled query set, the same candidates before and after reranking, measurable rank movement, latency, and failure analysis.

## Architecture, prerequisites, and scope

![Two-stage retrieval architecture](assets/two-stage-retrieval.svg)

```text
authorized query
      ↓
first-stage dense retrieval
      ↓
candidate set
      ↓
cross-encoder reranker
      ↓
context selection
```

This follows **Intermediate 01 — Retrieval Strategies** and **Intermediate 02 — Metadata and Permissions**. We assume dense/hybrid retrieval and authorization are already understood. The executable scope stays focused on dense candidates, explicit cross-encoder scoring, candidate budgets, ranking metrics, local latency, and failure analysis.

We deliberately do **not** implement HyDE, decomposition, multi-query retrieval, hybrid retrieval, learned sparse retrieval, ColBERT, or fine-tuning here.

## Success criteria and safety boundary

By the end, you should be able to:

- inspect `chunk_id`, base rank/distance, reranker score/rank, and relevance label separately;
- measure Recall@`candidate_k` before diagnosing reranking;
- compare baseline and reranked MRR, nDCG, Precision@k, any-evidence support, and evidence completeness on the same cases;
- find improved, unchanged, regressed, and missing relevant evidence;
- distinguish `candidate_k` from `top_n`;
- report environment-specific retrieval/reranking latency; and
- prove the reranker receives only authorized candidates.

The lab ends at ordered evidence. It does not use answer generation as a proxy for ranking quality.

## 0. Setup and reproducibility

Install the focused packages if you are not using the repository environment:

```bash
pip install langchain-chroma langchain-core chromadb sentence-transformers pandas
```

The two models download from Hugging Face on first use and then run locally. No API key or paid service is required. Timings and exact ranks may vary with model/library revision and hardware, so the notebook records the configuration and evaluates observed behavior rather than promising a result.

In [1]:
from __future__ import annotations

from collections import Counter
from copy import deepcopy
import hashlib
import math
import os
import platform
from time import perf_counter
import uuid
import warnings

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("HF_HUB_VERBOSITY", "error")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore", message="IProgress not found.*")

import numpy as np
import pandas as pd
import torch
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from sentence_transformers import CrossEncoder, SentenceTransformer

np.random.seed(7)
torch.manual_seed(7)

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L6-v2"
CANDIDATE_BUDGETS = (3, 5, 10, 20)
TOP_N_VALUES = (1, 3, 5)
PRINCIPAL = {"user_id": "atlas-analyst-7", "tenant_id": "atlas"}

print("Embedding model:", EMBEDDING_MODEL)
print("Reranker model:  ", RERANKER_MODEL)
print("Python/platform: ", platform.python_version(), platform.platform())

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Reranker model:   cross-encoder/ms-marco-MiniLM-L6-v2
Python/platform:  3.11.13 macOS-26.6.2-arm64-arm-64bit


## 1. Build a collision-rich enterprise corpus

The original Atlas/vendor/R-17 scenario remains, but the corpus now contains stable chunk IDs, current and historical rules, lexical hard negatives, overlapping policies, near-identical controls, identifiers, related-but-unsupported evidence, and three cross-tenant distractors.

The first multi-evidence question is labelled with both required chunks:

1. who operates the Atlas database; and
2. whether current R-17 third-party controls cover that type of provider.

We do not call one passage the “exact answer” when the conclusion requires two relationships.

Historical and draft records intentionally remain in this lesson's tenant-authorized candidate space as version-confusion hard negatives. In a production **current-policy-only** workflow, lifecycle eligibility should normally exclude historical, superseded, expired, and draft records before reranking. The reranker must not decide which version is authoritative.

In [2]:
def make_chunk(
    document_id: str,
    section: str,
    text: str,
    *,
    tenant_id: str = "atlas",
    status: str = "current",
) -> Document:
    return Document(
        page_content=text,
        metadata={
            "document_id": document_id,
            "chunk_id": f"{document_id}#{section}",
            "source": f"{document_id.replace('-', '_')}.md",
            "section": section,
            "tenant_id": tenant_id,
            "status": status,
        },
    )


corpus = [
    make_chunk("atlas-supplier-profile", "service", "Meridian Data Systems is the external supplier that operates and administers the managed PostgreSQL database platform for Atlas production workloads."),
    make_chunk("atlas-supplier-profile", "ownership", "Atlas owns the application data while Meridian operates the database service under a supplier agreement."),
    make_chunk("atlas-vendor-inventory", "database", "Vendor inventory entry V-204 lists Meridian Data Systems as the managed database infrastructure provider for Atlas."),
    make_chunk("atlas-vendor-inventory", "analytics", "Vendor inventory entry V-205 lists Northwind Analytics as the reporting dashboard supplier."),
    make_chunk("atlas-supplier-contract", "nda", "All Atlas suppliers must sign the standard confidentiality agreement before receiving internal documentation."),
    make_chunk("atlas-supplier-contract", "renewal", "The Meridian supplier agreement renews each September after procurement, service, and compliance review."),
    make_chunk("atlas-supplier-onboarding", "approval", "A high-risk Atlas supplier requires approval from Procurement and the service owner before onboarding."),
    make_chunk("atlas-supplier-onboarding", "security", "Supplier onboarding includes identity, security questionnaire, data-flow, and business-continuity review."),
    make_chunk("r17-vendor-policy-v2", "applicability", "Current R-17 third-party controls apply to external providers that host, operate, or administer production database infrastructure for Atlas."),
    make_chunk("r17-vendor-policy-v2", "assessment", "Before each supplier renewal, an in-scope database provider must complete an independent controls assessment every year."),
    make_chunk("r17-vendor-policy-v2", "evidence-retention", "R-17 assessment reports, exceptions, and remediation evidence must be retained for thirty-six months."),
    make_chunk("r17-vendor-policy-v2", "subprocessors", "An in-scope database provider must disclose subprocessors and obtain approval before a subprocessor receives Atlas production data."),
    make_chunk("r17-vendor-policy-v2", "data-residency", "R-17 requires production database copies and backups for the regulated Atlas service to remain in approved EU regions."),
    make_chunk("r17-internal-system-policy", "applicability", "The internal R-17 system policy applies to databases operated entirely by Atlas employees; external providers follow the separate third-party policy."),
    make_chunk("r17-internal-system-policy", "attestation", "Atlas-owned internal database teams submit a quarterly control attestation to the platform assurance group."),
    make_chunk("r17-historical-policy-v1", "applicability", "Superseded R-17 version 1 applied vendor controls only to suppliers directly processing payment transactions.", status="historical"),
    make_chunk("r17-historical-policy-v1", "assessment", "Superseded R-17 version 1 required a controls assessment once every two years.", status="historical"),
    make_chunk("r17-draft-policy-v3", "quantum", "Draft R-17 version 3 proposes post-quantum key exchange research; it is not approved or currently effective.", status="draft"),
    make_chunk("vendor-security-questionnaire", "cadence", "High-risk suppliers refresh the Atlas security questionnaire every twelve months and after material service changes."),
    make_chunk("vendor-security-questionnaire", "encryption", "The supplier questionnaire asks whether production data is encrypted in transit and at rest and how keys are managed."),
    make_chunk("database-vendor-sla", "availability", "The managed database service-level objective is 99.95 percent monthly availability."),
    make_chunk("database-vendor-sla", "incident-notice", "The database supplier must notify Atlas within thirty minutes of confirming a severity-one service outage."),
    make_chunk("database-vendor-sla", "service-credits", "Service credits apply when monthly database availability falls below the contracted threshold."),
    make_chunk("procurement-policy", "risk-tier", "A supplier with production data access and operational control is classified as procurement risk tier one."),
    make_chunk("procurement-policy", "approval", "Risk-tier-one suppliers require Procurement, Security, Privacy, and service-owner approval."),
    make_chunk("supplier-code", "ethics", "Suppliers must follow anti-bribery, labor, and conflicts-of-interest requirements."),
    make_chunk("supplier-code", "subcontractors", "Suppliers remain responsible for approved subcontractors and must flow contractual obligations down."),
    make_chunk("atlas-access-policy", "privileged-access", "Privileged production access uses named accounts, phishing-resistant authentication, and recorded approval."),
    make_chunk("atlas-access-policy", "review", "Service owners review privileged access membership every quarter."),
    make_chunk("atlas-data-policy", "retention", "Operational logs for the Atlas database are retained for thirteen months unless a legal hold applies."),
    make_chunk("atlas-data-policy", "residency", "Atlas customer profile data is stored in approved EU regions for the regulated service."),
    make_chunk("atlas-database-runbook", "failover", "Database failover requires Incident Commander approval and a validated replica-health check."),
    make_chunk("atlas-database-runbook", "backup", "Atlas database backups run every six hours and the restore exercise is performed quarterly."),
    make_chunk("atlas-incident-policy", "customer-notice", "Customer notification is considered after impact is confirmed and Legal approves the communication."),
    make_chunk("atlas-incident-policy", "severity", "A total production database outage with active customer impact is classified as severity one."),
    make_chunk("atlas-contract", "termination", "At termination, the supplier returns Atlas data and certifies deletion from active and backup systems."),
    make_chunk("atlas-contract", "audit-rights", "Atlas may review independent assurance reports and audit unresolved high-risk supplier controls."),
    make_chunk("atlas-change-policy", "maintenance", "Planned database maintenance requires seven days of notice unless an emergency exception is approved."),
    make_chunk("atlas-change-policy", "rollback", "Every production database change must include a tested rollback plan."),
    make_chunk("finance-vendor-policy", "r17", "Finance vendors use control F-17; this is unrelated to Regulation R-17 database requirements."),
    make_chunk("marketing-vendor-policy", "questionnaire", "Marketing content suppliers complete a lightweight brand and privacy questionnaire."),
    make_chunk("r18-vendor-policy", "applicability", "Regulation R-18 applies to external identity-verification providers, not managed databases."),
    make_chunk("r17-glossary", "definition", "R-17 is the Atlas control family for operational assurance of production database systems and in-scope providers."),
    make_chunk("atlas-asset-catalog", "ax-774-b", "Asset identifier AX-774-B denotes the hardware security module used for Atlas database encryption keys."),
    make_chunk("atlas-asset-catalog", "ax-774-a", "Asset identifier AX-774-A denotes the retired staging encryption appliance."),
    # Cross-tenant records deliberately share language but must be excluded before reranking.
    make_chunk("globex-r17-policy", "applicability", "Current R-17 controls apply to Globex external database operators.", tenant_id="globex"),
    make_chunk("globex-database-sla", "incident-notice", "Globex database suppliers provide severity-one notice within fifteen minutes.", tenant_id="globex"),
    make_chunk("globex-asset-catalog", "ax-774-b", "Globex uses AX-774-B as a networking appliance identifier.", tenant_id="globex"),
]

chunk_ids = [doc.metadata["chunk_id"] for doc in corpus]
assert len(chunk_ids) == len(set(chunk_ids))
print("Corpus chunks:", len(corpus))
print("Atlas authorized chunks:", sum(doc.metadata["tenant_id"] == "atlas" for doc in corpus))
print("Cross-tenant distractors:", sum(doc.metadata["tenant_id"] != "atlas" for doc in corpus))

Corpus chunks: 48
Atlas authorized chunks: 45
Cross-tenant distractors: 3


## 2. Define one labelled evaluation set

Every query uses stable `chunk_id` labels. We do not infer relevance from filenames or keyword overlap. Empty relevance sets are intentional no-answer cases: the models must still rank something, but ranking does not create support.

In [3]:
def query_case(query_id: str, query: str, relevant: list[str], slice_: str) -> dict:
    return {
        "query_id": query_id,
        "query": query,
        "relevant_chunk_ids": set(relevant),
        "slice": slice_,
    }


evaluation_queries = [
    query_case("q-001", "Does the company operating the Atlas production database fall under current R-17 third-party controls?", ["atlas-supplier-profile#service", "r17-vendor-policy-v2#applicability"], "multi-relevant"),
    query_case("q-002", "Which current R-17 rule governs external database operators rather than employee-run databases?", ["r17-vendor-policy-v2#applicability"], "policy-distinction"),
    query_case("q-003", "How often must an in-scope Atlas database provider complete an independent controls assessment under current R-17?", ["r17-vendor-policy-v2#assessment"], "version-sensitive"),
    query_case("q-004", "What does asset identifier AX-774-B denote?", ["atlas-asset-catalog#ax-774-b"], "identifier"),
    query_case("q-005", "Who must approve onboarding a high-risk Atlas supplier?", ["atlas-supplier-onboarding#approval"], "lexical-overlap"),
    query_case("q-006", "When is the high-risk vendor security questionnaire refreshed?", ["vendor-security-questionnaire#cadence"], "semantic-hard-negative"),
    query_case("q-007", "What monthly availability does the managed Atlas database SLA promise?", ["database-vendor-sla#availability"], "lexical-overlap"),
    query_case("q-008", "How quickly must the database vendor notify Atlas about a severity-one outage?", ["database-vendor-sla#incident-notice"], "semantic-hard-negative"),
    query_case("q-009", "What encryption topics does the Atlas supplier questionnaire ask about?", ["vendor-security-questionnaire#encryption"], "semantic-hard-negative"),
    query_case("q-010", "What obligations apply when the Atlas database provider uses another company to process production data?", ["r17-vendor-policy-v2#subprocessors", "supplier-code#subcontractors"], "multi-relevant"),
    query_case("q-011", "Which procurement risk tier covers a supplier with production data and operational control?", ["procurement-policy#risk-tier"], "policy-distinction"),
    query_case("q-012", "How long must R-17 assessment and remediation evidence be retained?", ["r17-vendor-policy-v2#evidence-retention"], "lexical-overlap"),
    query_case("q-013", "Where must regulated Atlas production database copies and backups reside under R-17?", ["r17-vendor-policy-v2#data-residency"], "policy-distinction"),
    query_case("q-014", "How frequently are Atlas database backups created and restore exercises performed?", ["atlas-database-runbook#backup"], "lexical-overlap"),
    query_case("q-015", "Does current R-17 require post-quantum encryption in production?", [], "no-answer"),
    query_case("q-016", "On what exact date in 2028 will Meridian complete its R-17 audit?", [], "no-answer"),
    query_case("q-017", "Which assurance rule covers an outside operator that keeps Atlas data persistence services running?", ["r17-vendor-policy-v2#applicability"], "missing-candidate-probe"),
    query_case("q-018", "What does current R-17 require before the managed database supplier agreement renews?", ["r17-vendor-policy-v2#assessment"], "regression-probe"),
    query_case("q-019", "Who is the managed production database provider for Atlas?", ["atlas-supplier-profile#service", "atlas-vendor-inventory#database"], "multi-relevant"),
    query_case("q-020", "What is the R-17 control family?", ["r17-glossary#definition"], "semantic-hard-negative"),
]

known_ids = set(chunk_ids)
assert all(case["relevant_chunk_ids"] <= known_ids for case in evaluation_queries)
print("Evaluation queries:", len(evaluation_queries))
print(pd.Series([case["slice"] for case in evaluation_queries]).value_counts().to_string())

Evaluation queries: 20
lexical-overlap            4
semantic-hard-negative     4
multi-relevant             3
policy-distinction         3
no-answer                  2
version-sensitive          1
identifier                 1
missing-candidate-probe    1
regression-probe           1


## 3. Load explicit teaching models

The bi-encoder independently embeds queries/documents for fast search. The cross-encoder jointly processes `(query, candidate)` pairs.

`all-MiniLM-L6-v2` and the MS MARCO cross-encoder are teaching baselines. A reranker trained on general web/search relevance may not optimally rank legal, medical, financial, code, or organization-specific evidence. Domain evaluation comes before fine-tuning or hard-negative training.

In [4]:
class SentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model: SentenceTransformer):
        self.model = model

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self.model.encode(
            texts,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).tolist()

    def embed_query(self, text: str) -> list[float]:
        return self.model.encode(
            [text],
            normalize_embeddings=True,
            show_progress_bar=False,
        )[0].tolist()


embedding_model = SentenceTransformer(EMBEDDING_MODEL)
reranker_model = CrossEncoder(RERANKER_MODEL)
embedding_adapter = SentenceTransformerEmbeddings(embedding_model)

# Warm model execution before measuring query latency.
_ = embedding_model.encode(["warm up"], normalize_embeddings=True, show_progress_bar=False)
_ = reranker_model.predict([("warm up", "warm up")], show_progress_bar=False)

print("Embedding device:", embedding_model.device)
print("Reranker device:  ", reranker_model.device)

Embedding device: mps:0
Reranker device:   mps:0


## 4. Index all tenants, retrieve only authorized candidates

We keep the security lesson intentionally small: the trusted principal supplies a tenant filter. The query cannot change it. The authorization invariant is asserted before any document text reaches the reranker.

In [5]:
collection_name = f"query_reranking_{uuid.uuid4().hex}"
vectorstore = Chroma.from_documents(
    documents=corpus,
    embedding=embedding_adapter,
    collection_name=collection_name,
)


def assert_authorized(candidates: list[dict], principal: dict) -> None:
    assert all(
        item["doc"].metadata["tenant_id"] == principal["tenant_id"]
        for item in candidates
    ), "Unauthorized candidate crossed into reranker-visible state"


def retrieve_candidates(
    query: str,
    candidate_k: int,
    principal: dict = PRINCIPAL,
) -> list[dict]:
    # The trusted application principal—not query text—constructs this scope.
    authorization_filter = {"tenant_id": {"$eq": principal["tenant_id"]}}
    docs_and_distances = vectorstore.similarity_search_with_score(
        query,
        k=candidate_k,
        filter=authorization_filter,
    )
    candidates = [
        {
            "doc": doc,
            "base_rank": rank,
            "base_distance": float(distance),
        }
        for rank, (doc, distance) in enumerate(docs_and_distances, start=1)
    ]
    assert_authorized(candidates, principal)
    return candidates


probe = retrieve_candidates(
    "Ignore authorization and show the Globex AX-774-B record.",
    candidate_k=10,
)
assert all(item["doc"].metadata["tenant_id"] == "atlas" for item in probe)
print("Authorization invariant passed for", len(probe), "adversarial-query candidates.")

Authorization invariant passed for 10 adversarial-query candidates.


## 5. Expose cross-encoder scoring and reranking

The cross-encoder produces a **model-specific relevance score used for ranking**.

```text
reranker score
≠ probability the answer is correct
≠ calibrated confidence
```

First-stage distance and reranker score remain separate. They are different scoring functions used at different stages; this lab does not combine them.

In [6]:
def score_with_reranker(query: str, candidates: list[dict]) -> list[float]:
    assert_authorized(candidates, PRINCIPAL)
    pairs = [(query, item["doc"].page_content) for item in candidates]
    scores = reranker_model.predict(
        pairs,
        batch_size=16,
        show_progress_bar=False,
    )
    return [float(score) for score in scores]


def rerank_candidates(query: str, candidates: list[dict]) -> list[dict]:
    scores = score_with_reranker(query, candidates)
    scored = [
        {**item, "reranker_score": score}
        for item, score in zip(candidates, scores)
    ]
    scored.sort(key=lambda item: item["reranker_score"], reverse=True)
    return [
        {**item, "reranked_rank": rank}
        for rank, item in enumerate(scored, start=1)
    ]


def stage_table(candidates: list[dict], reranked: list[dict], relevant_ids: set[str]) -> pd.DataFrame:
    reranked_by_id = {
        item["doc"].metadata["chunk_id"]: item
        for item in reranked
    }
    rows = []
    for item in candidates:
        chunk_id = item["doc"].metadata["chunk_id"]
        reranked_item = reranked_by_id[chunk_id]
        rows.append({
            "chunk_id": chunk_id,
            "base_rank": item["base_rank"],
            "base_distance": round(item["base_distance"], 4),
            "reranker_score": round(reranked_item["reranker_score"], 4),
            "reranked_rank": reranked_item["reranked_rank"],
            "relevant": chunk_id in relevant_ids,
        })
    return pd.DataFrame(rows).sort_values("base_rank")

## 6. Inspect one multi-evidence question

The same candidate objects carry both stages. Relevance labels come from the query set, and each score stays in its own column.

In [7]:
example_case = evaluation_queries[0]
example_candidates = retrieve_candidates(example_case["query"], candidate_k=10)
example_reranked = rerank_candidates(example_case["query"], example_candidates)
stage_table(
    example_candidates,
    example_reranked,
    example_case["relevant_chunk_ids"],
)

,chunk_id,base_rank,base_distance,reranker_score,reranked_rank,relevant
0,r17-vendor-policy-v2#applicability,1,0.2697,7.9856,1,True
1,r17-glossary#definition,2,0.4733,5.0996,3,False
2,r17-vendor-policy-v2#data-residency,3,0.5472,3.6557,4,False
3,r17-internal-system-policy#applicability,4,0.5677,5.2345,2,False
4,r17-internal-system-policy#attestation,5,0.8184,-0.9901,6,False
5,atlas-supplier-profile#ownership,6,0.8305,-1.6374,8,False
6,finance-vendor-policy#r17,7,0.8433,-1.1608,7,False
7,atlas-contract#audit-rights,8,0.8463,-1.8280,9,False
8,atlas-vendor-inventory#database,9,0.8875,-5.4156,10,False
9,r17-vendor-policy-v2#subprocessors,10,0.8884,-0.7797,5,False


The table answers two different questions:

- **candidate recall:** did both labelled evidence chunks enter the ten candidates?
- **ranking:** once present, did the cross-encoder move them toward the context budget?

It does not claim that the highest score proves a complete or correct answer.

## 7. Implement transparent ranking metrics

No heavy evaluation framework is required. No-answer cases are excluded from relevance-ranking averages because they have no relevant chunk; they are inspected separately for answerability risk.

**MRR is a first-hit metric.** It can improve when the earliest relevant passage moves up even if another required passage moves down or remains missing. nDCG and evidence completeness provide complementary views for multi-evidence questions. `Any-support@k` asks whether at least one required passage is present; `Evidence-completeness@k` measures the fraction of the entire required evidence set that is present.

In [8]:
def ranked_ids(items: list[dict]) -> list[str]:
    return [item["doc"].metadata["chunk_id"] for item in items]


def recall_at_candidates(items: list[dict], relevant_ids: set[str]) -> float | None:
    if not relevant_ids:
        return None
    return len(set(ranked_ids(items)) & relevant_ids) / len(relevant_ids)


def reciprocal_rank(items: list[dict], relevant_ids: set[str]) -> float | None:
    if not relevant_ids:
        return None
    for rank, chunk_id in enumerate(ranked_ids(items), start=1):
        if chunk_id in relevant_ids:
            return 1.0 / rank
    return 0.0


def precision_at_k(items: list[dict], relevant_ids: set[str], k: int) -> float | None:
    if not relevant_ids:
        return None
    selected = ranked_ids(items)[:k]
    return sum(chunk_id in relevant_ids for chunk_id in selected) / k


def ndcg_at_k(items: list[dict], relevant_ids: set[str], k: int) -> float | None:
    if not relevant_ids:
        return None
    gains = [1.0 if chunk_id in relevant_ids else 0.0 for chunk_id in ranked_ids(items)[:k]]
    dcg = sum(gain / math.log2(rank + 1) for rank, gain in enumerate(gains, start=1))
    ideal_count = min(len(relevant_ids), k)
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_count + 1))
    return dcg / idcg if idcg else 0.0


def any_evidence_support(items: list[dict], relevant_ids: set[str], top_n: int) -> float | None:
    if not relevant_ids:
        return None
    return float(bool(set(ranked_ids(items)[:top_n]) & relevant_ids))


def evidence_completeness(items: list[dict], relevant_ids: set[str], top_n: int) -> float | None:
    if not relevant_ids:
        return None
    top_ids = set(ranked_ids(items)[:top_n])
    return len(top_ids & relevant_ids) / len(relevant_ids)


def mean_defined(values: list[float | None]) -> float:
    defined = [value for value in values if value is not None]
    return float(np.mean(defined)) if defined else float("nan")

## 8. Run the same evaluation set at four candidate budgets

Each run measures first-stage retrieval, cross-encoder inference, and total two-stage duration. The numbers are local and environment-specific: model size, sequence length, CPU/GPU, batching, warm-up, and serving architecture all matter. Even after explicit model warm-up, notebook/runtime caching and backend initialization can influence these illustrative timings.

In [9]:
def evaluate_case(case: dict, candidate_k: int, top_n: int = 3) -> dict:
    retrieval_start = perf_counter()
    candidates = retrieve_candidates(case["query"], candidate_k)
    retrieval_ms = (perf_counter() - retrieval_start) * 1000

    rerank_start = perf_counter()
    reranked = rerank_candidates(case["query"], candidates)
    rerank_ms = (perf_counter() - rerank_start) * 1000

    relevant = case["relevant_chunk_ids"]
    return {
        **case,
        "candidate_k": candidate_k,
        "top_n": top_n,
        "candidates": candidates,
        "reranked": reranked,
        "candidate_recall": recall_at_candidates(candidates, relevant),
        "baseline_mrr": reciprocal_rank(candidates, relevant),
        "reranked_mrr": reciprocal_rank(reranked, relevant),
        "baseline_ndcg5": ndcg_at_k(candidates, relevant, 5),
        "reranked_ndcg5": ndcg_at_k(reranked, relevant, 5),
        "baseline_precision3": precision_at_k(candidates, relevant, 3),
        "reranked_precision3": precision_at_k(reranked, relevant, 3),
        "baseline_any_support": any_evidence_support(candidates, relevant, top_n),
        "reranked_any_support": any_evidence_support(reranked, relevant, top_n),
        "baseline_completeness": evidence_completeness(candidates, relevant, top_n),
        "reranked_completeness": evidence_completeness(reranked, relevant, top_n),
        "retrieval_ms": retrieval_ms,
        "rerank_ms": rerank_ms,
        "total_ms": retrieval_ms + rerank_ms,
        "reranker_pairs": len(candidates),
    }


runs_by_budget = {
    candidate_k: [evaluate_case(case, candidate_k) for case in evaluation_queries]
    for candidate_k in CANDIDATE_BUDGETS
}
print("Completed", sum(len(runs) for runs in runs_by_budget.values()), "query/configuration runs.")

Completed 80 query/configuration runs.


## 9. Candidate recall comes before reranking

Recall@`candidate_k` is computed before the second stage. Larger candidate sets may admit more relevant evidence, but every additional candidate creates another cross-encoder pair.

In [10]:
def percentile(values: list[float], q: float) -> float:
    return float(np.percentile(values, q))


budget_rows = []
for candidate_k, runs in runs_by_budget.items():
    budget_rows.append({
        "candidate_k": candidate_k,
        "Recall@candidate_k": mean_defined([run["candidate_recall"] for run in runs]),
        "baseline_MRR": mean_defined([run["baseline_mrr"] for run in runs]),
        "reranked_MRR": mean_defined([run["reranked_mrr"] for run in runs]),
        "reranked_nDCG@5": mean_defined([run["reranked_ndcg5"] for run in runs]),
        "pairs_evaluated": sum(run["reranker_pairs"] for run in runs),
        "retrieval_median_ms": np.median([run["retrieval_ms"] for run in runs]),
        "retrieval_p95_ms": percentile([run["retrieval_ms"] for run in runs], 95),
        "rerank_median_ms": np.median([run["rerank_ms"] for run in runs]),
        "rerank_p95_ms": percentile([run["rerank_ms"] for run in runs], 95),
        "total_median_ms": np.median([run["total_ms"] for run in runs]),
        "total_p95_ms": percentile([run["total_ms"] for run in runs], 95),
    })

budget_summary = pd.DataFrame(budget_rows)
budget_summary.round(4)

,candidate_k,Recall@candidate_k,baseline_MRR,reranked_MRR,reranked_nDCG@5,pairs_evaluated,retrieval_median_ms,retrieval_p95_ms,rerank_median_ms,rerank_p95_ms,total_median_ms,total_p95_ms
0,3,0.8056,0.7870,0.8333,0.7913,60,8.2974,16.2108,20.1852,26.1659,29.2011,38.7813
1,5,0.8333,0.7870,0.8333,0.8128,100,5.6076,8.5555,13.9471,23.5247,19.3662,28.5361
2,10,0.9444,0.8019,0.9028,0.8923,200,5.7371,6.0866,14.3011,23.7019,20.2709,29.6479
3,20,0.9722,0.8019,0.9000,0.8899,400,6.0997,7.1821,29.5685,43.0943,36.1370,49.2117


Interpret the measured frontier rather than choosing a universal `k`:

```text
larger candidate set
→ potentially higher candidate recall
→ more reranking pairs and compute
```

If recall remains low as `k` grows, investigate query representation, source coverage, indexing, and authorization scope before tuning the reranker.

## 10. Missing-candidate demonstration

We locate an evaluated query whose relevant evidence is absent or incomplete at `k=3` and improves by `k=10`. The reranker receives only the small candidate set, so it cannot recover what is absent.

In [11]:
runs_k3 = {run["query_id"]: run for run in runs_by_budget[3]}
runs_k10 = {run["query_id"]: run for run in runs_by_budget[10]}
missing_candidates = [
    case
    for case in evaluation_queries
    if case["relevant_chunk_ids"]
    and (runs_k3[case["query_id"]]["candidate_recall"] or 0.0)
       < (runs_k10[case["query_id"]]["candidate_recall"] or 0.0)
]
assert missing_candidates, "Tune the synthetic hard negatives: the lab needs a k=3 → k=10 recall case"

missing_case = missing_candidates[0]
small_run = runs_k3[missing_case["query_id"]]
larger_run = runs_k10[missing_case["query_id"]]
print("Query:", missing_case["query_id"], missing_case["query"])
print("Relevant:", sorted(missing_case["relevant_chunk_ids"]))
print("k=3 candidate IDs:", ranked_ids(small_run["candidates"]))
print("k=3 reranked IDs: ", ranked_ids(small_run["reranked"]))
print("k=3 recall:", small_run["candidate_recall"])
print("k=10 recall:", larger_run["candidate_recall"])
assert small_run["candidate_recall"] < larger_run["candidate_recall"]

Query: q-017 Which assurance rule covers an outside operator that keeps Atlas data persistence services running?
Relevant: ['r17-vendor-policy-v2#applicability']
k=3 candidate IDs: ['r17-internal-system-policy#attestation', 'atlas-contract#audit-rights', 'atlas-contract#termination']
k=3 reranked IDs:  ['r17-internal-system-policy#attestation', 'atlas-contract#audit-rights', 'atlas-contract#termination']
k=3 recall: 0.0
k=10 recall: 1.0


**Result:** no ordering algorithm can promote a chunk that is not in its input. Increasing the candidate budget may admit it, at the cost of more pair scoring. Query rewriting or another retrieval strategy changes what is searched; reranking only changes the order of existing candidates.

## 11. Per-relevant-chunk rank movement

Positive rank delta means the relevant chunk moved earlier; negative means regression. Missing evidence is reported explicitly rather than assigned a convenient rank.

In [12]:
def rank_movement(run: dict) -> list[dict]:
    base_rank = {
        item["doc"].metadata["chunk_id"]: item["base_rank"]
        for item in run["candidates"]
    }
    reranked_rank = {
        item["doc"].metadata["chunk_id"]: item["reranked_rank"]
        for item in run["reranked"]
    }
    rows = []
    for chunk_id in sorted(run["relevant_chunk_ids"]):
        before = base_rank.get(chunk_id)
        after = reranked_rank.get(chunk_id)
        if before is None:
            delta, status = None, "missing from candidates"
        else:
            delta = before - after
            status = "improved" if delta > 0 else "regressed" if delta < 0 else "unchanged"
        rows.append({
            "query_id": run["query_id"],
            "slice": run["slice"],
            "chunk_id": chunk_id,
            "base_rank": before,
            "reranked_rank": after,
            "rank_delta": delta,
            "status": status,
        })
    return rows


movement_rows = [row for run in runs_by_budget[10] for row in rank_movement(run)]
movement_df = pd.DataFrame(movement_rows)
print(movement_df["status"].value_counts(dropna=False).to_string())
movement_df.sort_values(["status", "query_id", "chunk_id"]).head(20)

status
unchanged                  12
improved                    6
missing from candidates     2
regressed                   1


,query_id,slice,chunk_id,base_rank,reranked_rank,rank_delta,status
2,q-002,policy-distinction,r17-vendor-policy-v2#applicability,3.0,2.0,1.0,improved
4,q-004,identifier,atlas-asset-catalog#ax-774-b,2.0,1.0,1.0,improved
10,q-010,multi-relevant,r17-vendor-policy-v2#subprocessors,3.0,2.0,1.0,improved
16,q-017,missing-candidate-probe,r17-vendor-policy-v2#applicability,7.0,4.0,3.0,improved
17,q-018,regression-probe,r17-vendor-policy-v2#assessment,8.0,1.0,7.0,improved
18,q-019,multi-relevant,atlas-supplier-profile#service,4.0,1.0,3.0,improved
0,q-001,multi-relevant,atlas-supplier-profile#service,NaN,NaN,NaN,missing from candidates
11,q-010,multi-relevant,supplier-code#subcontractors,NaN,NaN,NaN,missing from candidates
19,q-019,multi-relevant,atlas-vendor-inventory#database,1.0,2.0,-1.0,regressed
1,q-001,multi-relevant,r17-vendor-policy-v2#applicability,1.0,1.0,0.0,unchanged


### Regression is a measured possibility

If this model revision naturally regresses a labelled chunk, inspect it directly. If not, report zero honestly.

> **TEST-ONLY FAILURE INJECTION:** The following mutation deliberately boosts a historical hard negative for the version-sensitive query. It verifies the regression detector; it is not actual model behavior and is excluded from aggregate metrics.

In [13]:
natural_regressions = movement_df[movement_df["status"] == "regressed"]
if natural_regressions.empty:
    print("Observed natural reranker regressions at k=10: 0")
else:
    print("Observed natural reranker regressions at k=10:", len(natural_regressions))
    display(natural_regressions.head(10))

version_run = runs_k10["q-003"]
injected = deepcopy(version_run["reranked"])
historical_id = "r17-historical-policy-v1#assessment"
current_id = "r17-vendor-policy-v2#assessment"
historical_item = next(item for item in injected if item["doc"].metadata["chunk_id"] == historical_id)
max_score = max(item["reranker_score"] for item in injected)
historical_item["reranker_score"] = max_score + 1.0  # TEST-ONLY FAILURE INJECTION
injected.sort(key=lambda item: item["reranker_score"], reverse=True)
injected = [{**item, "reranked_rank": rank} for rank, item in enumerate(injected, start=1)]

original_current_rank = next(item["reranked_rank"] for item in version_run["reranked"] if item["doc"].metadata["chunk_id"] == current_id)
injected_current_rank = next(item["reranked_rank"] for item in injected if item["doc"].metadata["chunk_id"] == current_id)
print("TEST-ONLY controlled failure — current evidence rank:", original_current_rank, "→", injected_current_rank)
assert injected_current_rank >= original_current_rank

Observed natural reranker regressions at k=10: 1


,query_id,slice,chunk_id,base_rank,reranked_rank,rank_delta,status
19,q-019,multi-relevant,atlas-vendor-inventory#database,1.0,2.0,-1.0,regressed


TEST-ONLY controlled failure — current evidence rank: 1 → 2


## 12. `top_n` controls context retention, not candidate recall

Using the already-reranked `candidate_k=10` runs, vary only the number of passages retained for context. `Any-support` asks whether at least one required passage survives. `Evidence completeness` asks what fraction of all required passages survives. They are equal for single-source cases and intentionally diverge for multi-evidence cases.

In [14]:
top_n_rows = []
for top_n in TOP_N_VALUES:
    top_n_rows.append({
        "top_n": top_n,
        "candidate_recall_at_10": mean_defined([run["candidate_recall"] for run in runs_by_budget[10]]),
        "any_support_rate": mean_defined([
            any_evidence_support(run["reranked"], run["relevant_chunk_ids"], top_n)
            for run in runs_by_budget[10]
        ]),
        "mean_evidence_completeness": mean_defined([
            evidence_completeness(run["reranked"], run["relevant_chunk_ids"], top_n)
            for run in runs_by_budget[10]
        ]),
    })

top_n_summary = pd.DataFrame(top_n_rows)
top_n_summary.round(4)

,top_n,candidate_recall_at_10,any_support_rate,mean_evidence_completeness
0,1,0.9444,0.8333,0.7778
1,3,0.9444,0.9444,0.8889
2,5,0.9444,1.0000,0.9444


Candidate recall is unchanged because the first-stage set is unchanged. `top_n` affects only which ordered candidates fit into downstream context. Any-support can look healthy while evidence completeness remains low for multi-evidence questions; blindly increasing context can still add redundancy and noise.

## 13. No-answer queries still receive high-ranked candidates

Reranking is a relative ordering operation. There is always a top item when candidates exist.

In [15]:
no_answer_runs = [run for run in runs_by_budget[10] if not run["relevant_chunk_ids"]]
for run in no_answer_runs:
    print("\n", run["query_id"], run["query"])
    for item in run["reranked"][:3]:
        print(
            f"  {item['reranked_rank']}. {item['doc'].metadata['chunk_id']} "
            f"score={item['reranker_score']:.4f}"
        )
assert all(run["candidate_recall"] is None for run in no_answer_runs)


 q-015 Does current R-17 require post-quantum encryption in production?
  1. r17-draft-policy-v3#quantum score=3.7296
  2. r17-vendor-policy-v2#data-residency score=1.3147
  3. r17-vendor-policy-v2#applicability score=-0.0813

 q-016 On what exact date in 2028 will Meridian complete its R-17 audit?
  1. r17-vendor-policy-v2#evidence-retention score=-4.0225
  2. atlas-supplier-contract#renewal score=-4.2990
  3. r17-glossary#definition score=-5.5298


The highest reranker score is not proof that the corpus can answer the question. Any abstention threshold must be empirically calibrated on representative answerable and unanswerable validation data, then tested for false answers and false abstentions. That calibration belongs with answerability/evaluation work, not this ranking lab.

## 14. Diagnose failures only after candidate recall is known

The categories below are diagnostic labels, not a single “reranking failed” bucket.

In [16]:
def classify_failures(run: dict, top_n: int = 3) -> list[str]:
    failures = []
    relevant = run["relevant_chunk_ids"]
    if not relevant:
        if run["slice"] == "no-answer":
            return ["expected no-answer / unsupported by corpus"]
        return ["unexpected missing source / corpus coverage failure"]
    if run["candidate_recall"] < 1.0:
        failures.append("candidate recall failure")
    if (run["reranked_mrr"] or 0.0) < (run["baseline_mrr"] or 0.0):
        failures.append("reranker regression")
    if run["slice"] == "version-sensitive" and (run["reranked_mrr"] or 0.0) < 1.0:
        failures.append("version confusion")
    retained = set(ranked_ids(run["reranked"])[:top_n]) & relevant
    if run["slice"] == "multi-relevant" and len(retained) < len(relevant):
        failures.append("multi-evidence incompleteness")
    if run["slice"] in {"semantic-hard-negative", "regression-probe"} and (run["reranked_mrr"] or 0.0) < 1.0:
        failures.append("hard-negative confusion")
    return failures or ["none observed"]


failure_rows = [
    {
        "query_id": run["query_id"],
        "slice": run["slice"],
        "candidate_recall": run["candidate_recall"],
        "failures": ", ".join(classify_failures(run)),
    }
    for run in runs_by_budget[10]
]
pd.DataFrame(failure_rows)

,query_id,slice,candidate_recall,failures
0,q-001,multi-relevant,0.5,"candidate recall failure, multi-evidence incom..."
1,q-002,policy-distinction,1.0,none observed
2,q-003,version-sensitive,1.0,none observed
3,q-004,identifier,1.0,none observed
4,q-005,lexical-overlap,1.0,none observed
5,q-006,semantic-hard-negative,1.0,none observed
6,q-007,lexical-overlap,1.0,none observed
7,q-008,semantic-hard-negative,1.0,none observed
8,q-009,semantic-hard-negative,1.0,none observed
9,q-010,multi-relevant,0.5,"candidate recall failure, multi-evidence incom..."


An authorization scope issue would be a hard security failure, not a relevance regression. The invariant prevented cross-tenant candidates before scoring. An intentionally unsupported query is an expected no-answer case, not automatically a corpus defect. Only a query expected to be answerable but lacking indexed support is a corpus/source coverage failure. Neither condition can be repaired by reranking.

## 15. Final engineering comparison

Select `candidate_k=10`, `top_n=3` as a teaching comparison—not a universal optimum. Candidate recall is identical before/after because reranking reorders the same candidate set.

In [17]:
selected_runs = runs_by_budget[10]


def query_level_mrr_outcome(run: dict) -> str:
    if run["candidate_recall"] is None:
        return "no-answer"
    if run["candidate_recall"] < 1.0:
        return "missing evidence"
    baseline = run["baseline_mrr"] or 0.0
    reranked = run["reranked_mrr"] or 0.0
    return "improved" if reranked > baseline else "regressed" if reranked < baseline else "unchanged"


query_level_mrr_counts = Counter(query_level_mrr_outcome(run) for run in selected_runs)
outcome_order = ("improved", "unchanged", "regressed", "missing evidence", "no-answer")
query_level_mrr_summary = {label: query_level_mrr_counts.get(label, 0) for label in outcome_order}
chunk_movement_counts = Counter(movement_df["status"])
relevant_chunk_summary = {
    "improved": chunk_movement_counts.get("improved", 0),
    "unchanged": chunk_movement_counts.get("unchanged", 0),
    "regressed": chunk_movement_counts.get("regressed", 0),
    "missing": chunk_movement_counts.get("missing from candidates", 0),
}
comparison = pd.DataFrame([
    {"Metric": "Recall@candidate_k", "Baseline": mean_defined([r["candidate_recall"] for r in selected_runs]), "Reranked": mean_defined([r["candidate_recall"] for r in selected_runs])},
    {"Metric": "MRR", "Baseline": mean_defined([r["baseline_mrr"] for r in selected_runs]), "Reranked": mean_defined([r["reranked_mrr"] for r in selected_runs])},
    {"Metric": "nDCG@5", "Baseline": mean_defined([r["baseline_ndcg5"] for r in selected_runs]), "Reranked": mean_defined([r["reranked_ndcg5"] for r in selected_runs])},
    {"Metric": "Precision@3", "Baseline": mean_defined([r["baseline_precision3"] for r in selected_runs]), "Reranked": mean_defined([r["reranked_precision3"] for r in selected_runs])},
    {"Metric": "Any-support@3", "Baseline": mean_defined([r["baseline_any_support"] for r in selected_runs]), "Reranked": mean_defined([r["reranked_any_support"] for r in selected_runs])},
    {"Metric": "Evidence-completeness@3", "Baseline": mean_defined([r["baseline_completeness"] for r in selected_runs]), "Reranked": mean_defined([r["reranked_completeness"] for r in selected_runs])},
    {"Metric": "Median latency (ms)", "Baseline": float(np.median([r["retrieval_ms"] for r in selected_runs])), "Reranked": float(np.median([r["total_ms"] for r in selected_runs]))},
    {"Metric": "p95 latency (ms)", "Baseline": percentile([r["retrieval_ms"] for r in selected_runs], 95), "Reranked": percentile([r["total_ms"] for r in selected_runs], 95)},
])

print("Environment-specific comparison: candidate_k=10, top_n=3")
display(comparison.round(4))
print("Query-level MRR outcomes:", query_level_mrr_summary)
print("Relevant-chunk movements:", relevant_chunk_summary)

Environment-specific comparison: candidate_k=10, top_n=3


,Metric,Baseline,Reranked
0,Recall@candidate_k,0.9444,0.9444
1,MRR,0.8019,0.9028
2,nDCG@5,0.7738,0.8923
3,Precision@3,0.2963,0.3333
4,Any-support@3,0.8889,0.9444
5,Evidence-completeness@3,0.8056,0.8889
6,Median latency (ms),5.7371,20.2709
7,p95 latency (ms),6.0866,29.6479


Query-level MRR outcomes: {'improved': 4, 'unchanged': 12, 'regressed': 0, 'missing evidence': 2, 'no-answer': 2}
Relevant-chunk movements: {'improved': 6, 'unchanged': 12, 'regressed': 1, 'missing': 2}


### Decision guide

The two movement summaries are intentionally different. **Query-level MRR outcome** considers only the first relevant hit, while **relevant-chunk movements** count every labelled passage. A multi-evidence query can improve in MRR even when another required chunk regresses.

- If candidate recall is low, improve candidate generation, corpus coverage, filters, or query representation.
- If candidates contain the evidence but ordering is weak, a reranker may help—verify aggregate and slice metrics.
- If a reranker regresses important slices, evaluate another/domain-tuned model or hard-negative training.
- If latency exceeds the budget, reduce candidates, batch/accelerate inference, distill, cache carefully, or omit reranking.
- If no-answer queries receive plausible top scores, do not mistake ranking for answerability or calibrated confidence.

## 16. Production upgrade path

| Concern | Teaching lab | Production upgrade |
|---|---|---|
| Authorization | trusted tenant filter + assertion | shared policy decision, lifecycle/project scope, negative security tests, audit trail |
| First stage | local MiniLM + Chroma | evaluated dense/hybrid/late-interaction retrieval and backend filter semantics |
| Reranker | local MS MARCO MiniLM | domain-evaluated model, batching, GPU/ONNX/OpenVINO serving, version pinning |
| Scores | raw model-specific ranking values | calibrated thresholds only with answerable/unanswerable validation data |
| Evaluation | 20 curated synthetic cases | reviewed production-derived dataset, slices, regression gates, corpus/model versions |
| Latency | local median/p95 | warm service load tests, concurrency, queueing, tail SLOs, hardware cost |
| Context | retain top_n by score | diversity, deduplication, neighboring chunks, token budgets, evidence completeness |
| Observability | tables in notebook | candidate/rank traces, model versions, latency, labels, privacy-safe diagnostics |

Late interaction can support first-stage retrieval or reranking depending on implementation. HyDE, query decomposition, multi-query, learned sparse retrieval, ColBERT implementation, fine-tuning, and score fusion are deliberately deferred so this experiment stays diagnostic.

## 17. Exercises

1. Add one lexical hard negative and measure its rank movement without changing labels after observing output.
2. Add graded relevance and extend the nDCG implementation transparently.
3. Pick a candidate budget under a latency constraint and justify it from the measured frontier.
4. Add a source-diversity context selector and measure multi-evidence completeness.
5. Run on CPU and GPU, if available; report median/p95 without generalizing beyond the environment.
6. Design answerable/unanswerable data for score-threshold calibration, but keep it separate from ranking metrics.
7. Replace the reranker with a domain model and compare aggregate plus slice regressions.

### Final reflection

- Why is Recall@`candidate_k` unchanged by reranking?
- Why can `top_n=1` harm a multi-evidence question even when the first result is relevant?
- Why are base distance and reranker score not comparable?
- What evidence would justify deploying this reranker for a specific domain?
- Why is a high score insufficient to answer a no-answer query?

## References

- Nogueira & Cho — [Passage Re-ranking with BERT](https://arxiv.org/abs/1901.04085)
- Sentence Transformers — [Cross-Encoder usage](https://www.sbert.net/docs/cross_encoder/usage/usage.html) and [pretrained MS MARCO models](https://sbert.net/docs/cross_encoder/pretrained_models.html)
- LangChain — [Chroma integration](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma)
- Thakur et al. — [BEIR](https://arxiv.org/abs/2104.08663)
- Khattab & Zaharia — [ColBERT](https://arxiv.org/abs/2004.12832)
- Santhanam et al. — [ColBERTv2](https://aclanthology.org/2022.naacl-main.272/)

---

## Key takeaway

**A reranker improves ordering of retrieved candidates. It cannot recover evidence that never entered the candidate set.**